# Exercise 5.3: Coverage with toy experiments

From *Programming in High Energy Particle Physics*, Chapter 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch05/ex05_3_solution.ipynb)

Exercise 5.3 Coverage with toy experiments Generate pseudo-experiments (toy MC) to estimate the coverage of your confidence intervals. For 1000 toy experiments with known signal yield, verify that the 68% (95%) confidence intervals contain the true value approximately 68% (95%) of the time.

<details><summary>Hint</summary>

In a loop, generate a dataset from the model with fixed true yields ( model.generate(mass, RooFit::Extended()) draws a Poisson total), fit it, and check whether \(|\hat{n}_\text{sig} - n_\text{true}| &lt; \sigma_{\hat{n}}\) (or \(1.96\,\sigma_{\hat{n}}\)). Reset the parameters to their starting values before each fit and use PrintLevel(-1) . With 1000 toys the statistical precision on a coverage of 68% is \(\sqrt{0.68\cdot0.32/1000} \approx 1.5\%\).

</details>

### Colab setup

The first run takes about 3–5 minutes and restarts the runtime once. After restart, run all cells from the top again. The next cell installs ROOT from conda-forge only if it is missing.

In [ ]:
import sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and importlib.util.find_spec("ROOT") is None:
    %pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
import importlib.util
if IN_COLAB and importlib.util.find_spec("ROOT") is None:
    !mamba install -q -y -c conda-forge root
import ROOT
ROOT.gROOT.SetBatch(True)
print("ROOT",ROOT.gROOT.GetVersion())

### Fit pseudo-experiments

Here the Gaussian mean and width and the exponential slope are fixed to their generating values; the exercise tests uncertainty coverage for the signal yield. Set `nToys = 1000` for the full study; 300 is a quick Colab default.

In [ ]:
%%writefile ex05_3.C
#include <RooAddPdf.h>
#include <RooDataSet.h>
#include <RooExponential.h>
#include <RooFitResult.h>
#include <RooGaussian.h>
#include <RooRandom.h>
#include <RooMsgService.h>
#include <RooRealVar.h>
#include <TH1D.h>
#include <TCanvas.h>
#include <memory>
#include <iostream>
void ex05_3() {
  using namespace RooFit;
  RooRandom::randomGenerator()->SetSeed(42);
  RooMsgService::instance().setGlobalKillBelow(RooFit::ERROR);
  RooRealVar mass("mass","mass",100,160,"GeV");
  RooRealVar mean("mean","peak",125); mean.setConstant(true);
  RooRealVar sigma("sigma","resolution",2,0.1,10); sigma.setConstant(true);
  RooGaussian signal("signal","signal",mass,mean,sigma);
  RooRealVar lambda("lambda","slope",-0.035); lambda.setConstant(true);
  RooExponential background("background","background",mass,lambda);
  const double trueSig=120,trueBkg=200;
  RooRealVar nsig("nsig","signal yield",trueSig,0,500);
  RooRealVar nbkg("nbkg","background yield",trueBkg,0,700);
  RooAddPdf model("model","S+B",RooArgList(signal,background),RooArgList(nsig,nbkg));
  const int nToys=300; // reduce for a quick preview
  int covered1=0,covered2=0,good=0;
  for(int i=0;i<nToys;++i) {
    nsig.setVal(trueSig);nbkg.setVal(trueBkg);
    std::unique_ptr<RooDataSet> data{model.generate(mass,Extended())};
    nsig.setVal(trueSig);nbkg.setVal(trueBkg);
    std::unique_ptr<RooFitResult> fit{model.fitTo(*data,Save(),Extended(true),PrintLevel(-1),Strategy(0))};
    if(fit->status()!=0 || fit->covQual()<2 || nsig.getError()<=0) continue;
    ++good;
    const double pull=std::abs(nsig.getVal()-trueSig)/nsig.getError();
    covered1+=(pull<1.0); covered2+=(pull<2.0);
  }
  std::cout << "valid fits=" << good << "/" << nToys << std::endl;
  std::cout << "68.3% interval coverage=" << double(covered1)/good << std::endl;
  std::cout << "95.4% interval coverage=" << double(covered2)/good << std::endl;
  TH1D h("h","Toy interval coverage;Interval;Coverage [%]",2,0,2);
  h.GetXaxis()->SetBinLabel(1,"#pm1#sigma");h.GetXaxis()->SetBinLabel(2,"#pm2#sigma");
  h.SetBinContent(1,100.0*covered1/good);h.SetBinContent(2,100.0*covered2/good);
  h.SetMinimum(0);h.SetMaximum(100);
  TCanvas c("c","coverage",650,450); h.Draw("BAR2");c.SaveAs("ex05_3_coverage.pdf");
}


In [ ]:
!root -l -b -q ex05_3.C

For a well-behaved, large-yield fit, the book expects approximately 68.3% within ±1σ and 95.4% within ±2σ. In 1,000 toys the binomial uncertainties are about 1.5 and 0.7 percentage points. Small yields and the nonnegative yield boundary can spoil this approximation.